### Handwritten Digit Classification using CNN & RNN

In [12]:
import pandas as pd 
import numpy as np 
import torch 
import torch.nn as nn 
import torch.optim as optim

### Loading the dataset

In [13]:
import torchvision
from torchvision import datasets,transforms
from torchvision.datasets import MNIST


In [14]:
transform=transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.1307),(0.3081))
    ]
)

In [15]:
trainset=MNIST(root="./data",transform=transform, train=True,download=True)
testset=MNIST(root="./data",transform=transform,train=False,download=True)

### Datasets and Dataloaders 

In [16]:
from torch.utils.data import DataLoader

In [17]:
trainset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.1307, std=0.3081)
           )

In [18]:
testset

Dataset MNIST
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.1307, std=0.3081)
           )

In [19]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

### Defining RNN Model

In [20]:
class RNN(nn.Module):
    def __init__(self,input_size=28,hidden_size=128,num_layers=1,num_classes=10):
        super().__init__()


        self.hidden_size=hidden_size
        self.num_layers=num_layers


        # RNN layer

        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        # Fully connected layer

        self.fc=nn.Linear(hidden_size,num_classes)
    
    def forward(self,x):
        # x.shape => (batch_size,1,28,28)

        x=x.squeeze(1)
        
        h0=torch.zeros(
            self.num_layers,
            x.size(0),
            self.hidden_size
        )

        # RNN Forward

        out,hidden=self.rnn(x,h0)

        out=out[:,-1,:]

        out=self.fc(out)

        return out



In [21]:
model=RNN()

In [22]:
print(model)

RNN(
  (rnn): RNN(28, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)


In [23]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

### Training the Model 

In [25]:
epochs = 10 

for epoch in range(epochs):
    model.train()

    training_loss=0.0

    for images,labels in trainloader:

        optimizer.zero_grad()

        outputs=model(images)

        loss=criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        training_loss+=loss.item()


    print(f"{epoch+1}/{epochs},loss={training_loss/len(trainloader)}")





1/10,loss=0.1384610954864717
2/10,loss=0.12853522451151647
3/10,loss=0.12385807122522866
4/10,loss=0.12165761733207224
5/10,loss=0.11694241383585578
6/10,loss=0.10607994488303436
7/10,loss=0.10197627071444113
8/10,loss=0.10239786833429387
9/10,loss=0.10328728120092914
10/10,loss=0.1037971436538533


In [27]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in testloader:

        outputs = model(images)

        # Predicted class
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 96.87%


In [28]:
print(accuracy)

96.87


### Now using CNN 

In [29]:
import torch
import torch.nn as nn 
import torch.optim as optim

### Defining the CNN model

In [30]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(in_channels=1,out_channels=32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),

            nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),

            nn.Conv2d(in_channels=64,out_channels=128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)

        )

        self.fc_layers=nn.Sequential(
            nn.Linear(128*3*3,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x 

In [31]:
model=CNN()

In [32]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

### Training the CNN model

In [33]:
epochs=10

for epoch in range(epochs):

    cnn_training_loss=0.0

    model.train()

    for images,labels in trainloader:

        optimizer.zero_grad()

        outputs=model(images)

        loss=criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        cnn_training_loss+=loss.item()

    print(f"{epoch+1}/{epochs},loss => {cnn_training_loss/len(trainloader)}")

        

1/10,loss => 0.14798532901202707
2/10,loss => 0.04280473606862393
3/10,loss => 0.03168819449869405
4/10,loss => 0.02273540010068641
5/10,loss => 0.019599939244404025
6/10,loss => 0.015442853515840297
7/10,loss => 0.011710668624699907
8/10,loss => 0.012242568551959158
9/10,loss => 0.011782771329289982
10/10,loss => 0.007647804671792937


### Evaluating the CNN model 

In [37]:
model.eval()

total=0
correct=0

for images,labels in testloader:

    outputs=model.forward(images)

    _,predicted=torch.max(outputs,1)

    correct+=(labels==predicted).sum().item()

    total+=labels.size(0)

accuracy=correct/total*100

In [38]:
print("Accuracy => ",accuracy)

Accuracy =>  99.16
